# When the exchange closes: microstructure in stock perps

Every number in `post.md` comes from a cell below. Data: the public Aperiodic Python client (`pip install "aperiodic[polars]"`, key in `APERIODIC_API_KEY`).

In [1]:
import json, datetime as dt, polars as pl
import data, study
from model import *
ROOT = __import__('pathlib').Path('~/closed_market').expanduser()
data.download(ROOT)   # cached after the first run

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
panel = pl.concat([data.bars(ROOT, t) for t in data.UNIVERSE], how='diagonal_relaxed')
df = features(panel, data.INDEX)
s = df.filter(pl.col('time').dt.date() < study.END)['session'].value_counts(normalize=True).sort('session')
print(df['sym'].n_unique(), 'symbols'); print(s); print('closed share', s.filter(pl.col('session').is_in(CLOSED))['proportion'].sum())

28 symbols
shape: (6, 2)
┌────────────┬────────────┐
│ session    ┆ proportion │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ afterhours ┆ 0.115291   │
│ holiday    ┆ 0.019332   │
│ overnight  ┆ 0.232524   │
│ premarket  ┆ 0.158703   │
│ regular    ┆ 0.187601   │
│ weekend    ┆ 0.286549   │
└────────────┴────────────┘
closed share 0.5384044510110483


## Trade prices bounce; mid prices do not

In [3]:
study.bounce(panel)   # (mean IC over symbols, t across symbols, symbols)

{'last': (-0.0519362417096025, -5.321787650024695, 28),
 'mid': (-0.040078950753943285, -9.389439952167738, 28)}

## Each feature alone, by session

In [4]:
feat = study.single_features(df); feat

{'regular': {'book_imbalance': (0.008153540644646524, 2.728186211122107, 28),
  'premium': (-0.015870438922359044, -4.913537725012372, 28),
  'return_15m': (-0.0011601907248885422, -0.41113709523860403, 28),
  'index_return_15m': (-0.0012646510909866359, -0.47185709667989006, 28)},
 'overnight': {'book_imbalance': (0.022159188590545133, 7.909014208805548, 28),
  'premium': (-0.03518406183210414, -8.932065702774649, 28),
  'return_15m': (-0.003931393813265558, -1.4834700062850843, 28),
  'index_return_15m': (0.015799637191350302, 3.942212785662842, 28)},
 'weekend': {'book_imbalance': (0.06267392946743947, 12.25681298493906, 28),
  'premium': (-0.03905756347085992, -6.852242090787693, 28),
  'return_15m': (-0.04196305768287702, -9.565336354439621, 28),
  'index_return_15m': (0.036596633866802696, 5.219489554423185, 28)}}

In [5]:
study.binance_gap(ROOT, df)

{'symbols': 26,
 'closed': (-0.0325518356159097, -11.689892072276503, 26),
 'regular': (-0.005688829863171558, -1.5716503789038847, 26)}

In [6]:
study.median_usd = study.depth(df); study.median_usd

999.9726799999999

## The four-feature model, walked forward

In [7]:
closed = walk_forward(df, CLOSED, study.TEST_MONTHS, Model())
regular = walk_forward(df, REGULAR, study.TEST_MONTHS, Model())
print('rank IC closed', rank_ic(closed), 'regular', rank_ic(regular))
{m: rank_ic(closed.filter(pl.col('time').dt.month() == m)) for m in study.TEST_MONTHS}

rank IC closed 0.051727600871906226 regular -0.0027029582257178403


{4: 0.07500233227320288,
 5: 0.08536821000748909,
 6: 0.057491715909887345,
 7: 0.04179454147423809,
 8: 0.05597673436956833,
 9: 0.04736828516477998,
 10: 0.03270863575182508,
 11: 0.024318278977422628}

In [8]:
tc, tr = backtest(closed), backtest(regular)
rows = {'closed': tc, 'regular': tr, 'closed 3bp': backtest(closed, fee=3.0), 'closed 5m late': backtest(closed, delay_bars=1),
        'closed 3bp + late': backtest(closed, fee=3.0, delay_bars=1), 'closed 4.5bp': backtest(closed, fee=4.5)}
pl.DataFrame([{'run': k, **study.halves(v)} for k, v in rows.items()])

run,trades,net_bp,t,sharpe,sharpe_apr_jul,sharpe_aug_nov
str,f64,f64,f64,f64,f64,f64
"""closed""",366.0,11.001385,2.541293,3.207253,2.126761,4.198913
"""regular""",358.0,-42.50317,-2.095265,-2.219997,-3.566589,-2.615004
"""closed 3bp""",366.0,8.001385,1.8483,2.355589,1.06179,3.487482
"""closed 5m late""",364.0,3.583442,1.091908,1.391591,0.096297,2.45014
"""closed 3bp + late""",364.0,0.583442,0.17778,0.226666,-1.339977,1.450744
"""closed 4.5bp""",366.0,5.001385,1.155308,1.478862,-0.004452,2.738175


In [9]:
print(tc.group_by('session').agg(trades=pl.len(), net_bp=pl.col('net_bp').mean()).sort('session'))
tc.group_by(m=pl.col('time').dt.month()).agg(trades=pl.len(), net_bp=pl.col('net_bp').mean()).sort('m')

shape: (3, 3)
┌───────────┬────────┬───────────┐
│ session   ┆ trades ┆ net_bp    │
│ ---       ┆ ---    ┆ ---       │
│ str       ┆ u32    ┆ f64       │
╞═══════════╪════════╪═══════════╡
│ holiday   ┆ 19     ┆ 36.271537 │
│ overnight ┆ 175    ┆ 7.132709  │
│ weekend   ┆ 172    ┆ 12.146067 │
└───────────┴────────┴───────────┘


m,trades,net_bp
i8,u32,f64
4,39,-5.627032
5,55,8.674605
6,64,-4.588359
7,34,34.870343
8,18,-7.381983
9,34,29.997063
10,48,23.809962
11,74,11.446197


## The simpler alternatives

In [10]:
study.ols(df)

{'ic': 0.048414938016300105,
 'abs_forecast_p999': 25.850500871502295,
 'abs_forecast_max': 100.40092626385862,
 'at_6bp': {'trades': 1588.0,
  'net_bp': 0.7894727153178377,
  't': 0.32369313354206125,
  'sharpe': 0.40720917071990687,
  'sharpe_apr_jul': 0.1175724644753356,
  'sharpe_aug_nov': 0.7106037333392634}}

In [11]:
study.thresholds(df)

{'rate_0.001': {'trades': 192.0,
  'net_bp': 5.987425515313967,
  't': 1.2256394364232535,
  'sharpe': 2.1267608452390605},
 'rate_0.002': {'trades': 341.0,
  'net_bp': 5.564554071551942,
  't': 1.5724750887263723,
  'sharpe': 2.6938131728312356},
 'rate_0.003': {'trades': 471.0,
  'net_bp': 3.054806933965958,
  't': 0.8212558565452701,
  'sharpe': 1.4113100876085494},
 'rate_0.005': {'trades': 760.0,
  'net_bp': 1.2717591704668474,
  't': 0.4760606787783588,
  'sharpe': 0.8198291856275313},
 'fixed_8bp': {'trades': 469.0,
  'net_bp': 9.775683519562197,
  't': 2.43280340834779,
  'sharpe': 3.241469063350881,
  'sharpe_apr_jul': 3.1345237978337552,
  'sharpe_aug_nov': 3.535640973643126}}

In [12]:
study.capacity(df)

{'3x150': {'trades': 452.0,
  'net_bp': 9.812386551356775,
  't': 2.8738315620254045,
  'sharpe': 3.538903691321956},
 '3x300': {'trades': 407.0,
  'net_bp': 11.704702777157271,
  't': 3.217747970741717,
  'sharpe': 3.9765266125364276},
 '3x600': {'trades': 443.0,
  'net_bp': 14.141275657482208,
  't': 3.8395105282517417,
  'sharpe': 4.713053335852805},
 '7x150': {'trades': 366.0,
  'net_bp': 11.001385028322847,
  't': 2.541292917991132,
  'sharpe': 3.2072527672707114},
 '7x300': {'trades': 427.0,
  'net_bp': 10.928712997179192,
  't': 2.8926561310658796,
  'sharpe': 3.666860895864717},
 '7x600': {'trades': 618.0,
  'net_bp': 6.828080400879745,
  't': 2.2972422146571216,
  'sharpe': 3.019863337857921},
 '15x150': {'trades': 342.0,
  'net_bp': 7.863237163849103,
  't': 1.761156272183287,
  'sharpe': 2.31388833430364},
 '15x300': {'trades': 480.0,
  'net_bp': 7.367699395568688,
  't': 2.1750655623975153,
  'sharpe': 2.7930322935658642},
 '15x600': {'trades': 636.0,
  'net_bp': 3.55901865

## Asian underlyings

In [13]:
study.asia(ROOT, df)

{'both_closed_weekday': 0.10385266525561394,
 'weekend': 0.062286505269683684,
 'home_open': 0.02007729260981912,
 'us_open': 0.06267485909290209}

## Charts

In [14]:
study.charts(feat, tc, tr)